# Lesson 18: PyTorch — the toolbox behind modern AI 🔥
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Work with **tensors** (PyTorch's version of lists of numbers) on CPU and GPU
2. Let **autograd** calculate slopes (gradients) for you
3. Build a model with `nn.Module` and write the **standard training loop**
4. Use `Dataset` and `DataLoader` to feed data in batches

**⏱️ Timing:** tensors 15' | autograd 10' | nn.Module 10' | full project 25'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🧰 An analogy: a Lego set for neural networks
You've already used PyTorch in Lessons 4, 6, 8, 9 and 11, but mostly by copying recipes. Today we open the box and learn what each piece does:
| Piece | What it is | Lego analogy |
|---|---|---|
| `torch.tensor` | A grid of numbers (any number of dimensions) | The bricks |
| `autograd` | Automatic slope calculation | Instructions that build themselves backwards |
| `nn.Module` | A reusable model part | A pre-built sub-assembly (wheel, wing) |
| `optim` | Rules for updating weights | The hands that adjust the model |
| `DataLoader` | Serves data in small batches | A conveyor belt of bricks |

PyTorch was created at Meta (Facebook) AI Research and is now run by the independent PyTorch Foundation. Most AI research papers use it.

## Part 1: Tensors 🧱

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

a = torch.tensor([1.0, 2.0, 3.0])                 # 1-D tensor (a vector)
b = torch.tensor([[1, 2, 3], [4, 5, 6]])          # 2-D tensor (a matrix)
c = torch.zeros(2, 3, 4)                          # 3-D tensor, e.g. 2 tiny RGB images
print(a, a.shape)
print(b, b.shape)
print("c has shape", c.shape, "and", c.numel(), "numbers")

In [ ]:
x = torch.tensor([[1., 2.], [3., 4.]])
print("add:\n", x + 10)
print("multiply element by element:\n", x * x)
print("matrix multiply (the core of every neural network):\n", x @ x)
print("mean:", x.mean().item(), "| max:", x.max().item())
print("reshape to 1 row:", x.reshape(1, 4))

In [ ]:
# Moving to the GPU: same code, much faster for big tensors
device = "cuda" if torch.cuda.is_available() else "cpu"
big = torch.randn(4000, 4000)
import time
for d in ["cpu", device]:
    t = big.to(d); t @ t                            # warm-up run (the first GPU call is always slow)
    if d == "cuda": torch.cuda.synchronize()
    start = time.time()
    for _ in range(5): t @ t
    if d == "cuda": torch.cuda.synchronize()
    print(f"{d}: {time.time() - start:.2f}s for 5 big matrix multiplications")

### 🧩 Exercise 1
1. Make a tensor that represents **one 28×28 grayscale picture**. What shape should it have? What about a batch of 64 colour pictures of 32×32?
2. Create `torch.arange(12)` and reshape it into a 3×4 grid, then a 2×2×3 block.
3. How much faster was the GPU? Why are GPUs good at this? (Thousands of small cores doing multiplications at the same time.)

In [ ]:
# your code

## Part 2: Autograd — slopes for free 📐
In Lesson 4 we wrote the slope formula by hand: for loss = (w − 3)², slope = 2(w − 3). PyTorch can work it out **automatically** for any formula.

In [ ]:
w = torch.tensor(-4.0, requires_grad=True)     # "please track how this number is used"
loss = (w - 3) ** 2
loss.backward()                                  # work out the slope backwards through the formula
print("PyTorch's slope:", w.grad.item(), "| by hand: 2 × (−4 − 3) =", 2 * (-4 - 3))

In [ ]:
# Gradient descent with autograd — no slope formula needed!
w = torch.tensor(-4.0, requires_grad=True)
for step in range(30):
    loss = (w - 3) ** 2 + torch.sin(3 * w)        # a bumpier valley; we never write its slope
    loss.backward()
    with torch.no_grad():
        w -= 0.05 * w.grad
    w.grad.zero_()                                  # ⚠️ reset, otherwise slopes add up
print("ended at w =", round(w.item(), 3))

### 🧩 Exercise 2
Remove `w.grad.zero_()` and run again. What goes wrong? (This is a very common beginner bug!)

## Part 3: Models with `nn.Module` 🏗️
Every PyTorch model is a class with two parts: `__init__` (the parts) and `forward` (how data flows through them).

In [ ]:
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.layer1 = nn.Linear(n_in, n_hidden)
        self.layer2 = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        return self.layer2(x)

net = TinyNet(4, 8, 3)
print(net)
for name, p in net.named_parameters():
    print(f"{name:15} shape {tuple(p.shape)}")
print("output for one random input:", net(torch.randn(1, 4)))

## Part 4: A complete project — classify iris flowers 🌸
The famous **Iris** dataset: 150 flowers, 4 measurements each (petal and sepal length and width), 3 species. We'll use the **standard training loop** that almost every PyTorch project follows.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

iris = load_iris()
X = torch.tensor(iris.data, dtype=torch.float32)
y = torch.tensor(iris.target)
X = (X - X.mean(0)) / X.std(0)                       # scale each measurement to a similar range
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)
print("species:", list(iris.target_names), "| one batch shape:", next(iter(train_loader))[0].shape)

In [ ]:
torch.manual_seed(0)
model = TinyNet(4, 16, 3).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

history = []
for epoch in range(50):
    model.train()
    for xb, yb in train_loader:                      # ⭐ THE STANDARD TRAINING LOOP ⭐
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)                             # 1. forward pass
        loss = loss_fn(pred, yb)                     # 2. how wrong?
        optimizer.zero_grad()                        # 3. clear old slopes
        loss.backward()                              # 4. compute new slopes
        optimizer.step()                             # 5. nudge the weights
    model.eval()
    with torch.no_grad():
        acc = (model(X_te.to(device)).argmax(1).cpu() == y_te).float().mean().item()
    history.append(acc)

import matplotlib.pyplot as plt
plt.plot(history); plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.title(f"final accuracy {history[-1]:.0%}"); plt.show()

In [ ]:
# Use the trained model on a brand-new flower
new_flower = torch.tensor([[5.9, 3.0, 5.1, 1.8]])               # sepal length, sepal width, petal length, petal width (cm)
scaled = (new_flower - torch.tensor(iris.data, dtype=torch.float32).mean(0)) / torch.tensor(iris.data, dtype=torch.float32).std(0)
probs = torch.softmax(model(scaled.to(device)), dim=1)[0].cpu()
for name, p in zip(iris.target_names, probs):
    print(f"{name:12} {p:.0%}")

In [ ]:
# Save and load a model — how trained models are shared
torch.save(model.state_dict(), "iris_model.pt")
copy = TinyNet(4, 16, 3)
copy.load_state_dict(torch.load("iris_model.pt"))
print("Loaded! Same answer:", copy(scaled).argmax().item() == model(scaled.to(device)).argmax().item())

### 🧩 Exercise 3
1. Write the 5 steps of the training loop from memory on paper. You will see them in almost every PyTorch project.
2. Change `batch_size`, `lr` and the hidden size. Which change matters most?
3. Add a third layer to `TinyNet`.

### 🚀 Challenge
Rewrite the Lesson 3 digits classifier (`sklearn.datasets.load_digits`) using `TinyNet(64, 64, 10)` and the same training loop. Can you beat 97%?

## 📌 Summary
- **Tensors** = grids of numbers that can live on the GPU
- **Autograd** works out gradients for you with `.backward()`
- **nn.Module** = `__init__` (parts) + `forward` (data flow)
- The 5-step loop: **forward → loss → zero_grad → backward → step**